# Redis List vs Set — Find Performance Comparison

This notebook demonstrates the performance difference between searching for a value in a **Redis List** (`LPOS`) vs a **Redis Set** (`SISMEMBER`).

| Structure | Find Command | Time Complexity |
|-----------|-------------|------------------|
| List      | `LPOS`      | O(N) — scans every element |
| Set       | `SISMEMBER` | O(1) — hash lookup |

## 1. Connect to Redis

In [ ]:
import redis

r = redis.Redis(host='redis', port=6379, decode_responses=True)

print('Redis ping:', r.ping())
print('Redis version:', r.info('server')['redis_version'])

## 2. Generate 10 000 Random Items and Populate Redis

In [ ]:
import random
import string

DATASET_SIZE = 10_000
LIST_KEY = 'demo:mylist'
SET_KEY  = 'demo:myset'

def random_string(length=8):
    """Generate a random alphanumeric string."""
    return ''.join(random.choices(string.ascii_letters + string.digits, k=length))

# Build the dataset
random.seed(42)
items = [random_string() for _ in range(DATASET_SIZE)]

# Clean up any previous runs
r.delete(LIST_KEY, SET_KEY)

# Use pipelines for fast bulk inserts
pipe = r.pipeline()
for item in items:
    pipe.rpush(LIST_KEY, item)   # append to list
    pipe.sadd(SET_KEY,  item)   # add to set
pipe.execute()

print(f'List length : {r.llen(LIST_KEY):,}')
print(f'Set  cardinality: {r.scard(SET_KEY):,}')

## 3. Choose a Search Target

We pick **one value that exists** (worst-case for the list — it could be near the end) and **one value that does NOT exist** (always worst-case for the list).

In [ ]:
# A value guaranteed to exist (last element → worst case for LPOS)
existing_value = items[-1]

# A value guaranteed NOT to exist
missing_value = 'XXXXXXXX'

print('Existing value :', existing_value)
print('Missing  value :', missing_value)

## 4. Benchmark — Average Lookup Time over N Repetitions

In [ ]:
import time

N_REPS = 1_000

def benchmark(label, fn, reps=N_REPS):
    start = time.perf_counter()
    for _ in range(reps):
        fn()
    elapsed = time.perf_counter() - start
    total_ms = elapsed * 1000
    avg_ms   = total_ms / reps
    print(f'{label:<45}  avg: {avg_ms:.4f} ms   total: {total_ms:.1f} ms')
    return avg_ms, total_ms

print(f'Running {N_REPS:,} lookups each...\n')

# --- Existing value ---
print('=== Value EXISTS in the collection ===')
list_exist_avg, _  = benchmark('List  LPOS      (existing)',  lambda: r.lpos(LIST_KEY, existing_value))
set_exist_avg,  _  = benchmark('Set   SISMEMBER (existing)',  lambda: r.sismember(SET_KEY, existing_value))

print()

# --- Missing value ---
print('=== Value MISSING from the collection ===')
list_miss_avg, _   = benchmark('List  LPOS      (missing)',   lambda: r.lpos(LIST_KEY, missing_value))
set_miss_avg,  _   = benchmark('Set   SISMEMBER (missing)',   lambda: r.sismember(SET_KEY, missing_value))

## 5. What happens if we have more data?

Retry the previous steps with more items in your List and Set

## 6. Clean Up

In [ ]:
r.delete(LIST_KEY, SET_KEY)
print('Keys deleted. Redis is clean.')

---
## Key Takeaways

| | Redis List | Redis Set |
|---|---|---|
| **Find command** | `LPOS` | `SISMEMBER` |
| **Complexity** | **O(N)** — must scan each element | **O(1)** — hash table lookup |
| **Best case** | Value at index 0 | Always O(1) |
| **Worst case** | Value at end, or missing | Always O(1) |
| **Use when** | Order matters, duplicates allowed | Membership testing, uniqueness needed |

> **Rule of thumb:** If you only need to answer *"is X in the collection?"*, always prefer a **Set**.